# ⚡ Amazon ML Challenge 2026 — Kaggle Parallel Pipeline

This notebook is specifically designed for **Kaggle Notebooks (T4x2, 30GB RAM)**.
It maximizes performance by running **CPU workloads (Baseline Training)** and **GPU workloads (BGE-M3 Extraction)** in parallel simultaneously to save massive amounts of time!

## 🛠️ Step 1: System Hardware Check

In [ ]:
import psutil, os
import torch
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
cpus = os.cpu_count()
gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"KAGGLE HARDWARE: {cpus} CPU Cores | {ram_gb:.2f} GB RAM | {gpus} GPUs")

## 📦 Step 2: Clone Repo, Pull LFS, & Free Up 7.5GB Disk Space

In [ ]:
import os

# 1. Clean up Kaggle working directory in case of previous failed runs
!rm -rf /kaggle/working/AmazonMLChallenge2026

# 2. Install Git LFS and Clone
!apt-get install -y git-lfs > /dev/null
!git lfs install
!git clone https://github.com/priyanshu-ogdev/AmazonMLChallenge2026.git
%cd AmazonMLChallenge2026

# 3. Pull LFS files (temporarily spikes disk usage to ~11 GB)
!git lfs pull

# 4. IMMEDIATELY delete the .git folder (frees up ~5.3 GB of redundant LFS cache!)
!rm -rf .git

# 5. Recombine the split 2.4GB index and delete the parts (frees up another 2.4 GB!)
!cat output/phase1_blocking_train/candidate_index.pkl.part* > output/phase1_blocking_train/candidate_index.pkl
!rm output/phase1_blocking_train/candidate_index.pkl.part*

print("\n✅ Git LFS Pull and Workspace Optimization Complete!")
!df -h /kaggle/working


## ⚙️ Step 3: Install Dependencies

In [ ]:
!pip install -q -r code/business_entity_resolution/requirements.txt
!chmod +x scripts/*.sh
!bash scripts/00_verify_environment.sh --python-path python3

## 🚀 Step 4: PHASE 1 - Blocking & Lexical Features (Sequential)
This runs the memory-heavy Stage 1 Blocking. It will use up to 29GB of your 31GB RAM, finish entirely, and release the memory before extracting the Stage 2c lexical features.

In [ ]:
%%bash
set -e
export PYTHON_PATH="python3"
export DISABLE_LOGGING="true"

echo "=================================================="
echo " STEP 4.1: STAGE 1 BLOCKING"
echo "=================================================="
bash scripts/run_all_phases.sh --to-phase 1 --python-path python3

echo "=================================================="
echo " STEP 4.2: STAGE 2C LEXICAL FEATURES"
echo "=================================================="
# Extract lexical features, but skip BGE for now
bash scripts/02c_extract_pair_features.sh --candidate-file output/phase1_blocking_train/candidate_pairs.tsv --output-dir output/phase2_features_train --skip-bge
echo "✅ Stage 1 and Lexical Features COMPLETE!"


## ⚡ Step 5: PHASE 2 - Parallel CPU & GPU Workloads
This splits the execution into two parallel tracks:
1. **Track A (Background)**: Train the Baseline XGBoost Model on CPU.
2. **Track B (Foreground)**: Extract BGE-M3 and Qwen features on the T4x2 GPUs.

When both finish, the Final XGBoost Model will automatically be trained!

In [ ]:
%%bash
set -e
export PYTHON_PATH="python3"
export DISABLE_LOGGING="true"

echo "=================================================="
echo " SPAWNING PARALLEL WORKLOADS"
echo "=================================================="

# ---------------- TRACK A: BASELINE CPU ---------------- #
(
    echo "[TRACK A - CPU] Starting Baseline XGBoost Training..."
    bash scripts/03_train_scoring_gbm.sh --features-dir output/phase2_features_train --output-dir output/phase3_scoring_train --baseline-only
    bash scripts/04_inference_and_decision.sh --scoring-dir output/phase3_scoring_train --output-dir output/phase4_predictions
    
    echo "[TRACK A - CPU] Safely backing up Baseline..."
    mkdir -p /kaggle/working/baseline_backup
    cp -r output/phase3_scoring_train /kaggle/working/baseline_backup/
    cp output/phase4_predictions/submission.csv /kaggle/working/baseline_backup/baseline_submission.csv
    echo "[TRACK A - CPU] Baseline Training COMPLETE!"
) & 
TRACK_A_PID=$!

# ---------------- TRACK B: BGE-M3 GPU ---------------- #
(
    echo "[TRACK B - GPU] Starting BGE-M3 Multi-GPU Extraction..."
    # Skip lexical features, ONLY run BGE and Qwen extraction
    bash scripts/02c_extract_pair_features.sh --candidate-file output/phase1_blocking_train/candidate_pairs.tsv --output-dir output/phase2_features_train --skip-lexical --include-qwen
    echo "[TRACK B - GPU] Multi-GPU Extraction COMPLETE!"
) & 
TRACK_B_PID=$!

# Wait for both tracks to finish
echo "Waiting for parallel execution to finish..."
wait $TRACK_A_PID
wait $TRACK_B_PID
echo "✅ Parallel Workloads Finished!"

echo "=================================================="
echo " FINAL XGBOOST PIPELINE"
echo "=================================================="
bash scripts/03_train_scoring_gbm.sh --features-dir output/phase2_features_train --output-dir output/phase3_scoring_train
bash scripts/04_inference_and_decision.sh --scoring-dir output/phase3_scoring_train --output-dir output/phase4_predictions
echo "✅ Entire Pipeline Complete!"


## 📦 Step 6: Final Kaggle Output Generation

In [ ]:
import os

zip_name = "/kaggle/working/final_submission_artifacts.zip"
!zip -r -q {zip_name} output/ -x "output/**/_tmp_worker_chunks/*" "output/**/*.tmp"
print(f"Final package saved to {zip_name}. Download it from the right-hand Kaggle output panel!")